# Phase 0: Probing Latent Space Structure

Analysis notebook for motivating Experiment 2 (latent-space diffusion bridge).

Assumes `data/processed/probing/` exists with extracted encoder states from `extract_encoder_states.py`.

In [6]:
# add project root to path
import sys
from pathlib import Path

root = Path("/rds/general/user/tsv22/home/accent-robust-asr").resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [7]:
# imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import cosine
from scipy.stats import pearsonr, spearmanr

import umap

from src.config import (
    WHISPER_N_ENCODER_LAYERS,
    SPEAKER_L1,
    PROBE_PHONES,
    L1_GROUPS,
)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# device info

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

!nvidia-smi
# print physical and logical CPU cores
physical = psutil.cpu_count(logical=False)
logical = psutil.cpu_count(logical=True)
print(f"Physical cores: {physical}")
print(f"Logical cores (with hyperthreading): {logical}")

Using device: cuda
Wed May  6 18:58:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:CA:00.0 Off |                    0 |
|  0%   29C    P8             23W /  300W |       4MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------

## Load extracted embeddings

In [10]:
embeddings_dir = Path("/rds/general/user/tsv22/ephemeral/probing")
assert embeddings_dir.exists(), f"Missing: {embeddings_dir}"

from collections import defaultdict
import psutil
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

def load_speaker_embeddings(speaker_dir, split_dir):
    """Load all utterances and phone segments for a single speaker."""
    speaker = speaker_dir.name
    l1 = SPEAKER_L1.get(speaker, "Unknown")

    local_utts = []
    local_phones = []

    pt_files = sorted(speaker_dir.glob("*.pt"))
    for pt_file in pt_files:
        utt_id = pt_file.stem
        data = torch.load(pt_file, map_location="cpu")

        # Utterance-level: mean-pool over time
        layer_outputs = data["layer_outputs"]
        utterance_rep = layer_outputs.mean(dim=1)  # (12, 768)

        local_utts.append({
            "utt_id": utt_id,
            "speaker": speaker,
            "l1": l1,
            "split": split_dir,
            "layer_reps": utterance_rep.numpy(),
        })

        # Phone segments: FILTER to PROBE_PHONES only
        for phone_dict in data["phone_segments"]:
            if phone_dict["label"] in PROBE_PHONES:
                local_phones.append({
                    "label": phone_dict["label"],
                    "l1": phone_dict["l1"],
                    "speaker": phone_dict["speaker"],
                    "utt_id": utt_id,
                    "layer_reps": np.array(phone_dict["layer_reps"]),
                })

        del data, layer_outputs

    return speaker, local_utts, local_phones

# Collect all utterance-level and phone-segment embeddings
utterance_embeddings = []
phone_segment_data = []
speaker_utterance_indices = defaultdict(list)

process = psutil.Process(os.getpid())

print("Loading embeddings (8 parallel workers)...")
for split_dir in ["train", "dev", "test"]:
    split_path = embeddings_dir / split_dir
    if not split_path.exists():
        print(f"  Skipping {split_dir} (not found)")
        continue

    speaker_dirs = sorted([d for d in split_path.iterdir() if d.is_dir()])

    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {
            executor.submit(load_speaker_embeddings, sd, split_dir): sd.name
            for sd in speaker_dirs
        }

        for future in tqdm(as_completed(futures), total=len(futures), desc=f"  {split_dir}"):
            speaker, utts, phones = future.result()

            for utt in utts:
                utt_idx = len(utterance_embeddings)
                utterance_embeddings.append(utt)
                speaker_utterance_indices[speaker].append(utt_idx)

            phone_segment_data.extend(phones)

print(f"Loaded {len(utterance_embeddings)} utterances, {len(phone_segment_data)} probe phone segments")

current_mem = process.memory_info().rss / 1024 / 1024
print(f"Memory after loading: {current_mem:.1f} MB")

# Subsample: 100 utterances per speaker for visualization consistency
sampled_indices = []
for speaker, indices in speaker_utterance_indices.items():
    sampled = np.random.choice(indices, size=min(100, len(indices)), replace=False)
    sampled_indices.extend(sampled)

sampled_indices = np.array(sampled_indices)
print(f"Sampled {len(sampled_indices)} utterances for visualization (~{len(sampled_indices) // 7} per L1)")

Loading embeddings (8 parallel workers)...


  test: 100%|██████████| 7/7 [03:22<00:00, 28.96s/it] 

Loaded 31395 utterances, 239923 probe phone segments
Memory after loading: 12570.2 MB
Sampled 2800 utterances for visualization (~400 per L1)


In [11]:
sampled_utts = [utterance_embeddings[i] for i in sampled_indices]
print(f"Prepared sampled subset: {len(sampled_utts)} utterances")

Prepared sampled subset: 2800 utterances


## Fig 1: Layer probe accuracy curves

In [ ]:
print("Training probes...")

# Train/val split: use train+dev for training, test for evaluation
train_utts = [u for u in utterance_embeddings if u["split"] in ["train", "dev"]]
test_utts = [u for u in utterance_embeddings if u["split"] == "test"]

train_phone_segs = [p for p in phone_segment_data if any(u["utt_id"] == p["utt_id"] and u["split"] in ["train", "dev"] for u in utterance_embeddings)]
test_phone_segs = [p for p in phone_segment_data if any(u["utt_id"] == p["utt_id"] and u["split"] == "test" for u in utterance_embeddings)]

print(f"  Train utterances: {len(train_utts)}, test utterances: {len(test_utts)}")
print(f"  Train phone segs: {len(train_phone_segs)}, test phone segs: {len(test_phone_segs)}")

# Prepare data per layer
l1_acc_per_layer = []
phone_acc_per_layer = []

for layer_idx in tqdm(range(WHISPER_N_ENCODER_LAYERS), desc="  Probing layers"):
    # L1 classification on utterances
    X_train_utt = np.array([u["layer_reps"][layer_idx] for u in train_utts])  # (N, 768)
    y_train_utt = np.array([L1_GROUPS.index(u["l1"]) for u in train_utts])
    
    X_test_utt = np.array([u["layer_reps"][layer_idx] for u in test_utts])
    y_test_utt = np.array([L1_GROUPS.index(u["l1"]) for u in test_utts])
    
    scaler = StandardScaler()
    X_train_utt = scaler.fit_transform(X_train_utt)
    X_test_utt = scaler.transform(X_test_utt)
    
    clf_l1 = LogisticRegression(max_iter=1000, random_state=42)
    clf_l1.fit(X_train_utt, y_train_utt)
    l1_acc = clf_l1.score(X_test_utt, y_test_utt)
    l1_acc_per_layer.append(l1_acc)
    
    # Phoneme classification on phone segments
    X_train_phone = np.array([p["layer_reps"][layer_idx] for p in train_phone_segs])  # (N, 768)
    y_train_phone = np.array([p["label"] for p in train_phone_segs])
    
    X_test_phone = np.array([p["layer_reps"][layer_idx] for p in test_phone_segs])
    y_test_phone = np.array([p["label"] for p in test_phone_segs])
    
    # Map phone labels to indices
    phone_vocab = sorted(set(y_train_phone))
    phone_to_idx = {p: i for i, p in enumerate(phone_vocab)}
    y_train_phone_idx = np.array([phone_to_idx[p] for p in y_train_phone])
    y_test_phone_idx = np.array([phone_to_idx.get(p, -1) for p in y_test_phone])
    y_test_phone_idx = y_test_phone_idx[y_test_phone_idx >= 0]  # Filter unknowns
    X_test_phone = X_test_phone[y_test_phone_idx >= 0]
    
    scaler_phone = StandardScaler()
    X_train_phone = scaler_phone.fit_transform(X_train_phone)
    X_test_phone = scaler_phone.transform(X_test_phone)
    
    clf_phone = LogisticRegression(max_iter=1000, random_state=42)
    clf_phone.fit(X_train_phone, y_train_phone_idx)
    phone_acc = clf_phone.score(X_test_phone, y_test_phone_idx) if len(y_test_phone_idx) > 0 else 0
    phone_acc_per_layer.append(phone_acc)
    
    print(f"  Layer {layer_idx:2d}: L1={l1_acc:.3f}, Phone={phone_acc:.3f}")

print("\nDone!")

Training probes...


In [ ]:
# Plot layer probe accuracy
fig, ax = plt.subplots(figsize=(10, 6))

layers = np.arange(WHISPER_N_ENCODER_LAYERS)
ax.plot(layers, l1_acc_per_layer, marker='o', linewidth=2, markersize=8, label='L1 Classification (7-way)', color='#2E86AB')
ax.plot(layers, phone_acc_per_layer, marker='s', linewidth=2, markersize=8, label='Phoneme Classification (39-way)', color='#A23B72')

ax.set_xlabel('Encoder Layer', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Layer-wise Probe Accuracy: L1 vs Phoneme', fontsize=14, fontweight='bold')
ax.set_xticks(layers)
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='best')

plt.tight_layout()
output_path = Path("results/experiment2_phase0/figures/fig1_layer_probe_accuracy.png")
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"Saved: {output_path}")
plt.show()

# Identify target layer (where L1 accuracy peaks)
target_layer = np.argmax(l1_acc_per_layer)
print(f"\n=== TARGET LAYER: {target_layer} (L1 accuracy: {l1_acc_per_layer[target_layer]:.3f}) ===")

print(f"Computing UMAP for layer {target_layer}...")

# Panel (a): utterance-level embeddings — USE PRE-COMPUTED SAMPLED SUBSET
X_utt = np.array([u["layer_reps"][target_layer] for u in sampled_utts])  # (2.2k, 768)
y_utt_l1 = np.array([u["l1"] for u in sampled_utts])

scaler = StandardScaler()
X_utt_scaled = scaler.fit_transform(X_utt)

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
umap_utt = reducer.fit_transform(X_utt_scaled)

print(f"  Utterance UMAP shape: {umap_utt.shape}")

# Panel (b): phone segments for PROBE_PHONES only
probe_segs = [p for p in phone_segment_data if p["label"] in PROBE_PHONES]
X_phone = np.array([p["layer_reps"][target_layer] for p in probe_segs])  # (N, 768)
y_phone_l1 = np.array([p["l1"] for p in probe_segs])
y_phone_label = np.array([p["label"] for p in probe_segs])

X_phone_scaled = scaler.transform(X_phone)
umap_phone = reducer.transform(X_phone_scaled)

print(f"  Phone UMAP shape: {umap_phone.shape}")

In [ ]:
print(f"Computing UMAP for layer {target_layer}...")

# Panel (a): utterance-level embeddings — USE SAMPLED ONLY
sampled_utts = [utterance_embeddings[i] for i in sampled_indices]
X_utt = np.array([u["layer_reps"][target_layer] for u in sampled_utts])  # (2.2k, 768)
y_utt_l1 = np.array([u["l1"] for u in sampled_utts])

scaler = StandardScaler()
X_utt_scaled = scaler.fit_transform(X_utt)

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
umap_utt = reducer.fit_transform(X_utt_scaled)

print(f"  Utterance UMAP shape: {umap_utt.shape}")

# Panel (b): phone segments for PROBE_PHONES only
probe_segs = [p for p in phone_segment_data if p["label"] in PROBE_PHONES]
X_phone = np.array([p["layer_reps"][target_layer] for p in probe_segs])  # (N, 768)
y_phone_l1 = np.array([p["l1"] for p in probe_segs])
y_phone_label = np.array([p["label"] for p in probe_segs])

X_phone_scaled = scaler.transform(X_phone)
umap_phone = reducer.transform(X_phone_scaled)

print(f"  Phone UMAP shape: {umap_phone.shape}")

In [ ]:
# Plot UMAPs
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel (a): utterances colored by L1
l1_colors = {l1: plt.cm.tab10(i) for i, l1 in enumerate(L1_GROUPS)}
for l1 in L1_GROUPS:
    mask = y_utt_l1 == l1
    axes[0].scatter(umap_utt[mask, 0], umap_utt[mask, 1], label=l1, s=50, alpha=0.6, color=l1_colors[l1])

axes[0].set_xlabel('UMAP 1', fontsize=11)
axes[0].set_ylabel('UMAP 2', fontsize=11)
axes[0].set_title(f'Utterance-level Embeddings (Layer {target_layer}, colored by L1)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10, loc='best')
axes[0].grid(True, alpha=0.3)

# Panel (b): phone segments colored by L1
for l1 in L1_GROUPS:
    mask = y_phone_l1 == l1
    axes[1].scatter(umap_phone[mask, 0], umap_phone[mask, 1], label=l1, s=30, alpha=0.6, color=l1_colors[l1])

axes[1].set_xlabel('UMAP 1', fontsize=11)
axes[1].set_ylabel('UMAP 2', fontsize=11)
axes[1].set_title(f'Probe Phone Segments (Layer {target_layer}, colored by L1)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10, loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
output_path = Path("results/experiment2_phase0/figures/fig2_umap_latent_space.png")
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"Saved: {output_path}")
plt.show()

print("Computing paired displacement vs. WER...")

# Load test results to get WER per utterance
# Try to load from existing eval results; if not available, use placeholder
wer_data = {}
for results_csv in Path("results").rglob("*.csv"):
    try:
        df = pd.read_csv(results_csv)
        if "utterance_id" in df.columns and "utt_wer" in df.columns:
            for _, row in df.iterrows():
                wer_data[row["utterance_id"]] = row["utt_wer"]
    except Exception as e:
        pass

if not wer_data:
    print("  [WARN] No WER data found in results/. Using random placeholder WER.")
    np.random.seed(42)
    for u in utterance_embeddings:
        wer_data[u["utt_id"]] = np.random.uniform(0, 0.5)

# Match L2-ARCTIC test utterances to CMU-ARCTIC canonical by utterance ID (use SAMPLED)
l2_test_utts = [u for u in sampled_utts if u["split"] == "test" and u["l1"] != "English"]
cmu_utts = {u["utt_id"]: u for u in utterance_embeddings if u["l1"] == "English"}

displacements = []
wers = []
l1_labels = []

for l2_utt in l2_test_utts:
    # Try to find matching CMU utterance by utterance ID suffix (e.g., "ABA_arctic_a0001" -> "BDL_arctic_a0001")
    utt_suffix = "_".join(l2_utt["utt_id"].split("_")[1:])  # Remove speaker prefix
    matching_cmu = None
    for cmu_id, cmu_utt in cmu_utts.items():
        if utt_suffix in cmu_id:
            matching_cmu = cmu_utt
            break

    if matching_cmu is None:
        continue

    # Compute cosine distance at target layer
    l2_rep = l2_utt["layer_reps"][target_layer]  # (768,)
    cmu_rep = matching_cmu["layer_reps"][target_layer]  # (768,)

    dist = cosine(l2_rep, cmu_rep)
    wer = wer_data.get(l2_utt["utt_id"], np.nan)

    if not np.isnan(wer):
        displacements.append(dist)
        wers.append(wer)
        l1_labels.append(l2_utt["l1"])

displacements = np.array(displacements)
wers = np.array(wers)
l1_labels = np.array(l1_labels)

print(f"  Matched pairs: {len(displacements)}")

if len(displacements) > 0:
    r_pearson, p_pearson = pearsonr(displacements, wers)
    r_spearman, p_spearman = spearmanr(displacements, wers)
    print(f"  Pearson r = {r_pearson:.3f} (p={p_pearson:.3e})")
    print(f"  Spearman r = {r_spearman:.3f} (p={p_spearman:.3e})")

In [ ]:
print("Computing paired displacement vs. WER...")

# Load test results to get WER per utterance
# Try to load from existing eval results; if not available, use placeholder
wer_data = {}
for results_csv in Path("results").rglob("*.csv"):
    try:
        df = pd.read_csv(results_csv)
        if "utterance_id" in df.columns and "utt_wer" in df.columns:
            for _, row in df.iterrows():
                wer_data[row["utterance_id"]] = row["utt_wer"]
    except Exception as e:
        pass

if not wer_data:
    print("  [WARN] No WER data found in results/. Using random placeholder WER.")
    np.random.seed(42)
    for u in utterance_embeddings:
        wer_data[u["utt_id"]] = np.random.uniform(0, 0.5)

# Match L2-ARCTIC test utterances to CMU-ARCTIC canonical by utterance ID (use SAMPLED)
sampled_utts = [utterance_embeddings[i] for i in sampled_indices]
l2_test_utts = [u for u in sampled_utts if u["split"] == "test" and u["l1"] != "English"]
cmu_utts = {u["utt_id"]: u for u in utterance_embeddings if u["l1"] == "English"}

displacements = []
wers = []
l1_labels = []

for l2_utt in l2_test_utts:
    # Try to find matching CMU utterance by utterance ID suffix (e.g., "ABA_arctic_a0001" -> "BDL_arctic_a0001")
    utt_suffix = "_".join(l2_utt["utt_id"].split("_")[1:])  # Remove speaker prefix
    matching_cmu = None
    for cmu_id, cmu_utt in cmu_utts.items():
        if utt_suffix in cmu_id:
            matching_cmu = cmu_utt
            break

    if matching_cmu is None:
        continue

    # Compute cosine distance at target layer
    l2_rep = l2_utt["layer_reps"][target_layer]  # (768,)
    cmu_rep = matching_cmu["layer_reps"][target_layer]  # (768,)

    dist = cosine(l2_rep, cmu_rep)
    wer = wer_data.get(l2_utt["utt_id"], np.nan)

    if not np.isnan(wer):
        displacements.append(dist)
        wers.append(wer)
        l1_labels.append(l2_utt["l1"])

displacements = np.array(displacements)
wers = np.array(wers)
l1_labels = np.array(l1_labels)

print(f"  Matched pairs: {len(displacements)}")

if len(displacements) > 0:
    r_pearson, p_pearson = pearsonr(displacements, wers)
    r_spearman, p_spearman = spearmanr(displacements, wers)
    print(f"  Pearson r = {r_pearson:.3f} (p={p_pearson:.3e})")
    print(f"  Spearman r = {r_spearman:.3f} (p={p_spearman:.3e})")

In [ ]:
# Plot displacement vs. WER
if len(displacements) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Color by L1
    for l1 in sorted(set(l1_labels)):
        if l1 == "English":
            continue
        mask = l1_labels == l1
        ax.scatter(displacements[mask], wers[mask], label=l1, s=100, alpha=0.6)
    
    # Add regression line
    if len(displacements) > 1:
        z = np.polyfit(displacements, wers, 1)
        p = np.poly1d(z)
        x_line = np.linspace(displacements.min(), displacements.max(), 100)
        ax.plot(x_line, p(x_line), 'k--', linewidth=2, alpha=0.5, label='Linear fit')
    
    ax.set_xlabel('Cosine Distance (Accented vs. Canonical)', fontsize=12, fontweight='bold')
    ax.set_ylabel('WER', fontsize=12, fontweight='bold')
    ax.set_title(f'Latent Displacement vs. WER (Layer {target_layer})\nPearson r={r_pearson:.3f}, Spearman r={r_spearman:.3f}', 
                fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)
    
    plt.tight_layout()
    output_path = Path("results/experiment2_phase0/figures/fig3_displacement_vs_wer.png")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {output_path}")
    plt.show()
else:
    print("  No paired utterances found; skipping Fig 3.")

print("Computing displacement dimensionality...")

# Stack all (accented - canonical) displacement vectors (use SAMPLED)
l2_test_utts = [u for u in sampled_utts if u["split"] == "test" and u["l1"] != "English"]
cmu_utts = {u["utt_id"]: u for u in utterance_embeddings if u["l1"] == "English"}

displacement_vecs = []
displacement_l1s = []

for l2_utt in l2_test_utts:
    # Try to find matching CMU utterance
    utt_suffix = "_".join(l2_utt["utt_id"].split("_")[1:])
    matching_cmu = None
    for cmu_id, cmu_utt in cmu_utts.items():
        if utt_suffix in cmu_id:
            matching_cmu = cmu_utt
            break

    if matching_cmu is None:
        continue

    l2_rep = l2_utt["layer_reps"][target_layer]  # (768,)
    cmu_rep = matching_cmu["layer_reps"][target_layer]  # (768,)
    disp_vec = l2_rep - cmu_rep

    displacement_vecs.append(disp_vec)
    displacement_l1s.append(l2_utt["l1"])

displacement_vecs = np.array(displacement_vecs)  # (N, 768)
displacement_l1s = np.array(displacement_l1s)

print(f"  Displacement vectors: {displacement_vecs.shape}")

# PCA
pca = PCA()
pca.fit(displacement_vecs)

cumsum_var = np.cumsum(pca.explained_variance_ratio_)
n_components_95 = np.argmax(cumsum_var >= 0.95) + 1

print(f"  95% variance captured by {n_components_95} / {len(pca.components_)} components")

# Compute L1 clustering: within-L1 vs. cross-L1 cosine similarity
within_l1_sims = []
cross_l1_sims = []

unique_l1s = sorted(set(displacement_l1s))
for i, l1_i in enumerate(unique_l1s):
    for j, l1_j in enumerate(unique_l1s):
        mask_i = displacement_l1s == l1_i
        mask_j = displacement_l1s == l1_j
        vecs_i = displacement_vecs[mask_i]
        vecs_j = displacement_vecs[mask_j]

        if len(vecs_i) == 0 or len(vecs_j) == 0:
            continue

        # Cosine similarity between mean vectors
        mean_i = vecs_i.mean(axis=0)
        mean_j = vecs_j.mean(axis=0)
        sim = 1 - cosine(mean_i, mean_j)

        if i == j and len(vecs_i) > 1:
            within_l1_sims.append(sim)
        elif i < j:
            cross_l1_sims.append(sim)

print(f"  Within-L1 mean similarity: {np.mean(within_l1_sims):.3f}")
print(f"  Cross-L1 mean similarity: {np.mean(cross_l1_sims):.3f}")

In [ ]:
print("Computing displacement dimensionality...")

# Stack all (accented - canonical) displacement vectors (use SAMPLED)
sampled_utts = [utterance_embeddings[i] for i in sampled_indices]
l2_test_utts = [u for u in sampled_utts if u["split"] == "test" and u["l1"] != "English"]
cmu_utts = {u["utt_id"]: u for u in utterance_embeddings if u["l1"] == "English"}

displacement_vecs = []
displacement_l1s = []

for l2_utt in l2_test_utts:
    # Try to find matching CMU utterance
    utt_suffix = "_".join(l2_utt["utt_id"].split("_")[1:])
    matching_cmu = None
    for cmu_id, cmu_utt in cmu_utts.items():
        if utt_suffix in cmu_id:
            matching_cmu = cmu_utt
            break

    if matching_cmu is None:
        continue

    l2_rep = l2_utt["layer_reps"][target_layer]  # (768,)
    cmu_rep = matching_cmu["layer_reps"][target_layer]  # (768,)
    disp_vec = l2_rep - cmu_rep

    displacement_vecs.append(disp_vec)
    displacement_l1s.append(l2_utt["l1"])

displacement_vecs = np.array(displacement_vecs)  # (N, 768)
displacement_l1s = np.array(displacement_l1s)

print(f"  Displacement vectors: {displacement_vecs.shape}")

# PCA
pca = PCA()
pca.fit(displacement_vecs)

cumsum_var = np.cumsum(pca.explained_variance_ratio_)
n_components_95 = np.argmax(cumsum_var >= 0.95) + 1

print(f"  95% variance captured by {n_components_95} / {len(pca.components_)} components")

# Compute L1 clustering: within-L1 vs. cross-L1 cosine similarity
within_l1_sims = []
cross_l1_sims = []

unique_l1s = sorted(set(displacement_l1s))
for i, l1_i in enumerate(unique_l1s):
    for j, l1_j in enumerate(unique_l1s):
        mask_i = displacement_l1s == l1_i
        mask_j = displacement_l1s == l1_j
        vecs_i = displacement_vecs[mask_i]
        vecs_j = displacement_vecs[mask_j]

        if len(vecs_i) == 0 or len(vecs_j) == 0:
            continue

        # Cosine similarity between mean vectors
        mean_i = vecs_i.mean(axis=0)
        mean_j = vecs_j.mean(axis=0)
        sim = 1 - cosine(mean_i, mean_j)

        if i == j and len(vecs_i) > 1:
            within_l1_sims.append(sim)
        elif i < j:
            cross_l1_sims.append(sim)

print(f"  Within-L1 mean similarity: {np.mean(within_l1_sims):.3f}")
print(f"  Cross-L1 mean similarity: {np.mean(cross_l1_sims):.3f}")

In [ ]:
# Plot dimensionality
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel (a): PCA cumulative explained variance
n_comps_to_plot = min(100, len(pca.components_))
axes[0].plot(range(1, n_comps_to_plot + 1), cumsum_var[:n_comps_to_plot], marker='o', linewidth=2, markersize=6)
axes[0].axhline(y=0.95, color='r', linestyle='--', linewidth=2, label='95% variance')
axes[0].axvline(x=n_components_95, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Number of Components', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Cumulative Explained Variance', fontsize=11, fontweight='bold')
axes[0].set_title(f'PCA of Displacement Vectors\n(95% at {n_components_95} components)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Panel (b): within-L1 vs. cross-L1 similarity
sim_labels = ['Within-L1', 'Cross-L1']
sim_means = [np.mean(within_l1_sims), np.mean(cross_l1_sims)]
sim_stds = [np.std(within_l1_sims), np.std(cross_l1_sims)]
colors_bar = ['#2E86AB', '#A23B72']

axes[1].bar(sim_labels, sim_means, yerr=sim_stds, capsize=10, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Mean Cosine Similarity (of mean vectors)', fontsize=11, fontweight='bold')
axes[1].set_title('Displacement Vector Clustering by L1', fontsize=12, fontweight='bold')
axes[1].set_ylim([0, 1])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
output_path = Path("results/experiment2_phase0/figures/fig4_displacement_dimensionality.png")
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"Saved: {output_path}")
plt.show()

## Summary

In [ ]:
print("\n" + "="*60)
print("PHASE 0 SUMMARY")
print("="*60)
print(f"\nTarget Layer: {target_layer}")
print(f"  - L1 classification accuracy: {l1_acc_per_layer[target_layer]:.3f}")
print(f"  - Phoneme classification accuracy: {phone_acc_per_layer[target_layer]:.3f}")
print(f"\nL1 Clustering (Fig 2):")
print(f"  - Utterance-level embeddings show spatial structure")
print(f"  - Phone-segment embeddings: within-phone L1 displacement visible")
print(f"\nDisplacement Analysis (Fig 3):")
print(f"  - Paired utterances: {len(displacements)}")
if len(displacements) > 0:
    print(f"  - Pearson r (displacement vs. WER): {r_pearson:.3f}")
    print(f"  - Spearman r (displacement vs. WER): {r_spearman:.3f}")
print(f"\nDisplacement Dimensionality (Fig 4):")
print(f"  - Components for 95% variance: {n_components_95} / 768")
print(f"  - Within-L1 displacement similarity: {np.mean(within_l1_sims):.3f}")
print(f"  - Cross-L1 displacement similarity: {np.mean(cross_l1_sims):.3f}")
print(f"\nImplications for Bridge Design:")
if n_components_95 < 100:
    print(f"  ✓ Low-rank displacement: small bridge should suffice")
else:
    print(f"  ✗ Full-rank displacement: larger model may be needed")

if np.mean(within_l1_sims) > np.mean(cross_l1_sims):
    print(f"  ✓ L1-structured displacement: consider L1-conditioning")
else:
    print(f"  ✗ L1-agnostic displacement: universal bridge may suffice")

if len(displacements) > 0 and abs(r_pearson) > 0.3:
    print(f"  ✓ Displacement predicts WER: validates latent-space hypothesis")
else:
    print(f"  ? Weak displacement-WER correlation: investigate further")

print("\nFigures saved to: results/experiment2_phase0/figures/")
print("="*60)